# mT5-small Kaggle 训练并自动推送到 Hugging Face Hub

这份 notebook 支持：安装依赖、加载数据、训练 `mT5-small`、自动创建 Hugging Face 仓库并推送模型。

In [ ]:
%pip install -q transformers datasets sentencepiece accelerate huggingface_hub

In [ ]:
import json
from pathlib import Path

import torch
import transformers
import datasets
from datasets import Dataset, DatasetDict
from huggingface_hub import login, HfApi, create_repo
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# 这里填写你的 Hugging Face Token 和仓库名
# repo_id 的格式一般是: 用户名/仓库名
# 不存在的话，后面会自动创建

HF_TOKEN = "在这里填写你的_hf_token"
HF_REPO_ID = "在这里填写你的用户名/仓库名"

assert HF_TOKEN and HF_TOKEN != "在这里填写你的_hf_token", "请先填写 HF_TOKEN"
assert HF_REPO_ID and HF_REPO_ID != "在这里填写你的用户名/仓库名", "请先填写 HF_REPO_ID"

login(token=HF_TOKEN)
create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True, private=False)

print("Hugging Face 登录成功，仓库已就绪:", HF_REPO_ID)

In [ ]:
MODEL_NAME = "google/mt5-small"
OUTPUT_DIR = Path("./mt5_eda_output")

DATA_CANDIDATES = [
    Path("./datasets/eda_train_50000.jsonl"),
    Path("/kaggle/input/eda-generate/eda_train_50000.jsonl"),
    Path("/kaggle/input/eda-generate/datasets/eda_train_50000.jsonl"),
]

data_path = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError("没有找到 eda_train_50000.jsonl，请先确认 Kaggle 数据集挂载路径。")

print("model:", MODEL_NAME)
print("data path:", data_path)
print("output dir:", OUTPUT_DIR)

In [ ]:
# 这里的长度配置按你当前统计结果做保守设置

MAX_INPUT_LENGTH = 192
MAX_TARGET_LENGTH = 256
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACC_STEPS = 8
LEARNING_RATE = 5e-5
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
VAL_RATIO = 0.05
SAMPLE_LIMIT = None

In [ ]:
records = []
with data_path.open("r", encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        item = json.loads(line)
        records.append({
            "input_text": item["prompt"],
            "target_text": json.dumps(item["output"], ensure_ascii=False, separators=(",", ":"))
        })
        if SAMPLE_LIMIT is not None and len(records) >= SAMPLE_LIMIT:
            break

print("loaded samples:", len(records))
print(records[0]["input_text"])
print(records[0]["target_text"][:500])

In [ ]:
dataset = Dataset.from_list(records)
split_dataset = dataset.train_test_split(test_size=VAL_RATIO, seed=42)
dataset_dict = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})

print(dataset_dict)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

print("tokenizer and model ready")

In [ ]:
def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="tokenizing dataset",
)

print(tokenized_datasets)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("data collator ready")

In [ ]:
use_fp16 = torch.cuda.is_available()

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=use_fp16,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print(training_args)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("trainer ready")

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
# 先保存到本地输出目录

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("saved locally:", OUTPUT_DIR)

In [ ]:
# 自动推送到 Hugging Face Hub
# 如果仓库不存在，上面已经自动创建了

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

api = HfApi()
repo_info = api.repo_info(repo_id=HF_REPO_ID, token=HF_TOKEN)
print("push complete:", repo_info.id)

In [ ]:
# 简单推理测试

test_prompt = "设计一个完成RC低通滤波功能的 EDA 原理图，包含输入 VIN、电阻 R1、输出 OUT、电容 C1 和 接地 GND。请给出所有节点的坐标以及节点之间的连接关系。"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH)
inputs = {k: v.to(device) for k, v in inputs.items()}

generated_ids = model.generate(**inputs, max_length=MAX_TARGET_LENGTH)
result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(result)